In [2]:
# ============ RECALL Week 4 Day 3 -WARMUP ============
# 1.Why is LightGBM faster than XGBoost- two reasons :
  # Uses histogram based splits rather going intop each element
  # Leaf wise depth rather than level wise

# 2. What did the learning curve show as n_estimators grew?
  # It improves till a certian point and then plateaus
  # The training score kept improving toward 0 (perfect fit) while validation plateaued around 100-200 trees.
  #That gap between the two lines is the overfitting signature

# 3. RMSE in plain English : what does $28,809 actually mean
  # The mean error by LGBM model is off by $28,809

# =======================================

# **week4/ day 3**

###
-	Cross-validation: why the test set alone is not enough
-	K-fold CV from scratch : then replicate with cross_val_score
-	Stratified K-fold for imbalanced datasets: when and why


1. Fold 1: rows 0-79 train, rows 80-99 validate
2. Fold 2: rows 0-59 + 80-99 train, rows 60-79 validate
3. Fold 3: rows 0-39 + 60-99 train, rows 40-59 validate

- Each fold rotates the validation window by 20 rows. Every row gets to be in the validation set exactly once across all 5 folds.


1. A single split gives you one accuracy number that depends heavily on which 20% you happened to pick.
2. You could get lucky or unlucky with that split.
3. CV gives you 5 accuracy numbers and you average them, much more reliable estimate of true model performance.

## Task 1 : K-Fold from scratch
- Manually split data into K folds, train on K-1, validate on 1, rotate. Print accuracy for each fold and mean.


In [3]:
import pandas as pd
import numpy as np


df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

df.fillna({'Age': df['Age'].median()}, inplace=True)
df.dropna(subset=['Embarked'], inplace=True)
df['Title'] = df['Name'].str.extract(r', ([A-Za-z]+)\.')
df['AgeGroup'] = df['Age'].apply(lambda x: 'Child' if x<13 else 'Teen' if x<18 else 'Adult' if x<61 else 'Senior')
df['Sex_encoded'] = df['Sex'].map({'male':0, 'female':1})

In [4]:
features = ['Pclass', 'Age', 'Fare', 'Sex_encoded']
X = df[features]
y = df['Survived']

In [5]:
X.shape

(889, 4)

In [8]:
indices = np.arange(len(X))

folds = np.array_split(indices,5)

In [11]:
folds[0]

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177])

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
scores = []

for fold in folds:
  # 1. split into train and val indices
  val_idx = fold
  train_idx = np.concatenate([f for f in folds if f is not fold])

  ## 2. index into X and y
  X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
  y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

  # 3. fit model
  clf = DecisionTreeClassifier(random_state=42)
  clf.fit(X_train,y_train)

  # 4. predict and score
  y_pred = clf.predict(X_val)
  scores.append(accuracy_score(y_val,y_pred))

print(scores)
print(f"Mean CV score: {np.mean(scores):.3f}")

[0.7528089887640449, 0.7696629213483146, 0.8089887640449438, 0.7921348314606742, 0.8248587570621468]
Mean CV score: 0.790


Task 2 :Replicate with cross_val_score
1. Same data, same K : verify your scratch results match sklearn's output.


In [22]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

clf = DecisionTreeClassifier(random_state=42)
kf = KFold(n_splits=5, shuffle=False)

cv_score = cross_val_score(clf, X, y, cv=kf)

print(cv_score)
print(np.mean(cv_score))


[0.75280899 0.76966292 0.80898876 0.79213483 0.82485876]
0.7896908525360249



Task 3 : Why CV beats a single train/test split

1. Single train/test split : may give higher accuracy, but it trains on fixed data and trains on the single fixed mix.
2. CV: utlisises validation splits to train and test on different data throughout the loop.
3. Hence, CV catches patterns underlying in the data and hence performs well on test data o reven new data .


## Why Cross-Validation Beats a Single Train/Test Split
- A single train/test produces one accuracy score which heavily relies on the rows which happen to be in the test set. One can get lucky, unlucky with that particular split making it unreliable.

- K-fold CV rotates the validation window across all data, giving K-accuracy scores. Every row gets to be in Validation test exactily once. Averaging these scores produces a more reliable estimate of true model performance, that is less sensitive to any single split.

- - CV uses more overall data for training, reducing the variance of the performance estimate. Especially, on small datasets like Titanic (889 rows).  

Task 4 : Stratified K-Fold
1. Use StratifiedKFold on Titanic
2. compare fold-by-fold class distributions to regular KFold. When and why does it matter?


1. If one fold has 70% survivors it's an easier prediction problem than reality, and another fold with 20% survivors is harder.
2. accuracy scores vary wildly across folds not because of the model but because of the class distribution difference.

### Stratified KFold:
- fixes this by ensuring every fold has the same class ratio as the full dataset ~38% survived in every fold.

In [24]:
from sklearn.model_selection import StratifiedKFold, cross_val_score


skf = StratifiedKFold(n_splits=5, shuffle=False)
cv_score = cross_val_score(clf, X, y, cv=skf)

print(cv_score)
print(np.mean(cv_score))

[0.7247191  0.76404494 0.80898876 0.78651685 0.82485876]
0.781825683996699


In [25]:
# Regular KFold distributions
print("Regular KFold:")
for i, (_, val_idx) in enumerate(KFold(n_splits=5).split(X, y)):
    print(f"Fold {i+1}: {y.iloc[val_idx].value_counts(normalize=True).to_dict()}")

# Stratified distributions
print("\nStratified KFold:")
for i, (_, val_idx) in enumerate(StratifiedKFold(n_splits=5).split(X, y)):
    print(f"Fold {i+1}: {y.iloc[val_idx].value_counts(normalize=True).to_dict()}")

Regular KFold:
Fold 1: {0: 0.6741573033707865, 1: 0.3258426966292135}
Fold 2: {0: 0.5561797752808989, 1: 0.4438202247191011}
Fold 3: {0: 0.6123595505617978, 1: 0.38764044943820225}
Fold 4: {0: 0.5955056179775281, 1: 0.4044943820224719}
Fold 5: {0: 0.6497175141242938, 1: 0.3502824858757062}

Stratified KFold:
Fold 1: {0: 0.6179775280898876, 1: 0.38202247191011235}
Fold 2: {0: 0.6179775280898876, 1: 0.38202247191011235}
Fold 3: {0: 0.6179775280898876, 1: 0.38202247191011235}
Fold 4: {0: 0.6179775280898876, 1: 0.38202247191011235}
Fold 5: {0: 0.615819209039548, 1: 0.384180790960452}


## Regular KFold:
1.survived ratio varies from 32% to 44% across folds ->nearly 12% swing. That's significant.


## Stratified KFold:
1.every single fold sits at exactly 38.2% survived. Consistent across all 5 folds.

Task 5
- When would you use Stratified K-Fold over regular K-Fold?

## Stratified K-Fold over regular K-Fold
Use Stratified K-Fold when:
- Dataset has imbalanced classes -> e.g. Titanic (62% died, 38% survived)
- Regular KFold showed 12% swing in class ratio across folds (32%–44%),
  making accuracy estimates unreliable
- Stratified KFold guaranteed exactly 38.2% survivors in every fold,
  producing consistent and trustworthy CV scores

Rule of thumb: Always use StratifiedKFold for classification problems.
Only use regular KFold for regression (no classes to stratify).